#### Library imports

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import random
import scipy
import ipywidgets as widgets
import matplotlib.pyplot as plt
import sys

sys.path.append("/kaggle/input/datasets/alessandrotirelli/prw-utility-functions")
from eval_function import eval_search_prw, _compute_iou
from utils import fix_random, parse_annotations_file, draw_boxes, detect_pedestrians, crop_detections_from_frame, set_requires_grad_for_layer
from pathlib import Path
from typing import List, Optional, Callable
from torch.utils.data import Dataset
from torchvision.transforms import v2
from PIL import Image, ImageFont, ImageDraw
from ipywidgets import interact
from tqdm.auto import tqdm
from torchvision.models.detection.backbone_utils import resnet_fpn_backbone
from torchvision.ops.feature_pyramid_network import LastLevelP6P7
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection import RetinaNet
from torchvision.models import resnet50

#### Parameters definition

In [ ]:
data_root = Path("/kaggle/input/datasets/edoardomerli/prw-person-re-identification-in-the-wild") 

seed = 42 # random seed

detection_size = (128, 256) # widely used measures in the literature
crop_margin = 0

# detector
# NOTE: Detector hyperparameters (score_thresh and nms_thresh) are different from the ones used during training. 
#       This allows the model to reach a higher recall value.
detector_weights_path = Path("/kaggle/input/models/alessandrotirelli/first-stage-prw-detector/pytorch/default/1/retina_resnet34_ep_20_bs_4_lr_2e-05.pt")
score_thresh=0.20 # training: 0.05
nms_thresh=0.16  # training: 0.5

# embedder
embedder_weights_path = "/kaggle/input/models/alessandrotirelli/second-stage-prw-embedder/pytorch/default/23/resnet50_bnneck_pk16x16_emb2048_ep300.pt"
emb_size = 2048

#### GPU Initialization

In [11]:
device = "cpu"
if torch.cuda.is_available():
  print("All good, a Gpu is available.")
  device = torch.device("cuda:0")
else:
  print("Please set GPU via Edit -> Notebook Settings.")

All good, a Gpu is available.


#### Reproducibility and Deterministic Mode

In [ ]:
fix_random(seed)

# 1) Data Preparation

## Frame Dataset - Definition

In [13]:
class FrameDataset(Dataset):
    def __init__(
            self,
            data_root: Path,
            transforms: Optional[Callable] = None
        ) -> None:
        """Dataset representation of frames.

        Args:
            data_root: the path to the folder containing all data.
            transforms: the transformation to apply to the dataset.
        """
        
        # Define the expected file extensions and paths for images and annotations
        ext_images="jpg"
        ext_annotations="jpg.mat"
        path_images=data_root/"frames"
        path_annotations=data_root/"annotations"

        mat = scipy.io.loadmat(data_root/"frame_test.mat") # read Matlab file
        sample_indexes_list = [str(elem[0][0]) for elem in mat["img_index_test"]] # in the string format: 'cX_sY_ZZZZZZ'
        
        # Converting Python list to Python set to scale down access cost from O(n) to O(1) in list comprehension
        sample_indexes = set(sample_indexes_list)
        
        # Filters all images and annotations of the test split from the data folders
        self.image_paths = sorted([
            path for path in path_images.rglob(f"*.{ext_images}")
            if path.name.removesuffix(f".{ext_images}") in sample_indexes
        ])
        self.annotations = sorted([
            path for path in path_annotations.rglob(f"*.{ext_annotations}")
            if path.name.removesuffix(f".{ext_annotations}") in sample_indexes 
        ])

        self.transforms = transforms
        self.labels = ["__background__", "pedestrian"] # needed for RetinaNet

        # Sanity check on the loaded data
        if len(self.image_paths) != len(self.annotations):
            raise AssertionError(
                f"Images and annotations differs in size: {len(self.image_paths)} vs {len(self.annotations)}."
            )

    def __getitem__(self, idx):
        path_image = self.image_paths[idx]
        path_annotations = self.annotations[idx]
    
        # Image path -> PIL Image -> (optional) Transforms
        image = Image.open(path_image).convert("RGB")
        if self.transforms is not None:
            image = self.transforms(image)
    
        # Parse annotations
        pids, boxes = parse_annotations_file(path_annotations)
        
        # Handle the case of frames with no annotated pedestrians (i.e., no 'box_new' key in the annotation Matlab file) 
        # by creating empty tensors for boxes, labels and pids.
        if len(boxes) == 0: 
            boxes_xyxy = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros(0, dtype=torch.int64)
            pids = torch.zeros(0, dtype=torch.int64)
        else:
            boxes = torch.as_tensor(boxes, dtype=torch.float32)  # convert list of boxes to a Tensor of shape [N, 4]
            # Convert bounding box notation from [x, y, w, h] to [x_min, y_min, x_max, y_max] (required by RetinaNet)
            boxes_xyxy = torch.stack([
                boxes[:, 0],               # x_min = x
                boxes[:, 1],               # y_min = y
                boxes[:, 0] + boxes[:, 2], # x_max = x + w
                boxes[:, 1] + boxes[:, 3], # y_max = y + h
            ], dim=1)
                
            # RetinaNet requires a 'labels' key (1 = pedestrian, 0 = background)
            labels = torch.ones(len(pids), dtype=torch.int64)  # all boxes are pedestrians
            pids = torch.as_tensor(pids, dtype=torch.int64)
    
        target = {
            "boxes": boxes_xyxy,   # [N, 4] in xyxy format
            "labels": labels,      # [N] - required by RetinaNet
            "pids": pids           # [N] - useful for visualization putrposes
        }

        # Frame name in the format 'cX_sY_ZZZZZZ', useful for visualization purposes
        frame_name = path_image.stem
    
        return image, target, frame_name

    def __len__(self):
        return len(self.image_paths)

## Frame Dataset - Instantiation

In [14]:
# PIL/numpy to Image (subclass of torch.Tensor) + conversion & scaling to [0, 1]
frame_transforms = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True)
]) 

frame_dataset = FrameDataset(
    data_root=data_root,
    transforms=frame_transforms
)

labels = frame_dataset.labels
num_labels = len(labels) # required for RetinaNet

print(f"Number of frames: {len(frame_dataset)}")

Number of frames: 6112


## Frame Dataset - Visualization

In [15]:
@interact(
    sample_index = widgets.IntSlider(min=0, max=len(frame_dataset)-1, step=1, value=0, description="Sample Index")
)
def plot_image_and_boxes(
        sample_index: int
    ) -> None:
    """Plots an image from the training set with the respective bounding boxes.

    Args:
        sample_index: the index of the sampled image inside the training set.
    """

    image, target, frame_name = frame_dataset[sample_index]
    boxes = target["boxes"]
    pids = target["pids"]
    image = v2.ToPILImage()(image)
    
    cell_with_bb = draw_boxes(
        image,
        boxes=boxes,
        pids=pids,
        scores=[1.0] * len(boxes),
    )
    
    plt.figure(figsize=(10, 10))
    plt.imshow(cell_with_bb)
    plt.title(f"Frame: {frame_name}")
    plt.axis("off")
    plt.show()


interactive(children=(IntSlider(value=0, description='Sample Index', max=6111), Output()), _dom_classes=('widg…

# 2) First Stage - Detector Model

## Backbone instantiation

In [17]:
detector_backbone = resnet_fpn_backbone(
    backbone_name='resnet34', # ResNet34 as backbone
    weights='IMAGENET1K_V1', # pretrained on ImageNet-1k
    trainable_layers=5, # all ResNet layers trainable (stem + C2 to C5)
    returned_layers=[2, 3, 4], # last three layers of ResNet (C3, C4, C5)
    extra_blocks=LastLevelP6P7(256, 256) # add levels P6 and P7 to get the RetinaNet backbone as defined in the paper
)

# Using standard anchos scales
anchor_scales = tuple(2 ** scale for scale in [0, 1/3, 2/3])
# Using custom anchor scales and ratios to better fit pedestrian detection task 
# (exploiting dataset statistics given by the paper)
anchor_sizes = tuple(tuple(int(size * scale) for scale in anchor_scales) for size in [32, 64, 128, 256, 512]) # 5 sizes, one for each pyramid level (P3-P7)
anchor_ratios = ((1.5, 2.0, 3.0),) * len(anchor_sizes)  # Ratio = height / width (paper uses (0.5, 1.0, 2.0))

anchor_generator = AnchorGenerator(sizes=anchor_sizes, aspect_ratios=anchor_ratios) # anchor generator definition with custom anchor sizes and ratios

## RetinaNet Instantiation & Weights Loading

In [18]:
with tqdm(total=1, desc="Loading detector") as pbar:
    detector = RetinaNet(
        detector_backbone,
        num_labels,
        anchor_generator=anchor_generator,
        score_thresh=score_thresh,
        nms_thresh=nms_thresh,
        detections_per_img=100
    ).to(device)

    detector.load_state_dict(torch.load(detector_weights_path, weights_only=True))
    detector.eval() # put model in evaluation mode
    pbar.update(1)

Loading detector:   0%|          | 0/1 [00:00<?, ?it/s]

(Detector) Skipping training, using provided checkpoints.


# 3) Second Stage - Embedder Model

## Definition

In [19]:
class Embedder(nn.Module):
    """
        Neural network computing an image embedding.

        emb_size: the size for the embedding.
        normalize_emb: if True, normalizes the embedding using the L2 norm at eval time.
        pretrained_feat_ext: if True, uses a pretrained feature extractor.
        train_feat_ext: if True, trains the feature extractor.
    """
    def __init__(
            self,
            emb_size: int,
            normalize_emb: bool,
            pretrained_feat_ext: bool,
            train_feat_ext: bool
        ) -> None:
        
        # nn.Module initialization
        super().__init__()
        
        self.backbone = resnet50( # using resnet50 as widely adopted in the literature
            weights="IMAGENET1K_V2" if pretrained_feat_ext else None
        )
        feat_dim = self.backbone.fc.in_features
        self.backbone.fc = nn.Identity() # remove the final fully connected layer of ResNet
        set_requires_grad_for_layer(self.backbone, train_feat_ext)

        self.emb = nn.Linear(feat_dim, emb_size, bias=False) # embedding layer definition (fully connected layer without bias)
        # BatchNorm layer definition for the embedding (commonly referred to as "BNNeck" in the ReID literature) 
        # with bias term not trainable as suggested by the paper "Bag of Tricks and a Strong Baseline for Deep Person Re-identification"
        self.bnneck = nn.BatchNorm1d(emb_size)
        self.bnneck.bias.requires_grad_(False)

        self.normalize_emb = normalize_emb 

    def forward(self, x):
        feat = self.backbone(x)
        emb = self.emb(feat)
        y = self.bnneck(emb)

        if self.normalize_emb: # normalizing output embeddings 
            y = nn.functional.normalize(y)

        return y

## Instantiation & Weights loading

In [23]:
with tqdm(total=1, desc="Loading embedder") as pbar:
    embedder = Embedder(
        emb_size=emb_size,
        normalize_emb=True,
        pretrained_feat_ext=True,
        train_feat_ext=True
    ).to(device)

    embedder.load_state_dict(torch.load(embedder_weights_path, map_location=device))
    embedder = embedder.eval() # put model in evaluation mode
    pbar.update(1)

Loading embedder:   0%|          | 0/1 [00:00<?, ?it/s]

# 4) Running pipeline

### Preliminary step

In [26]:
detection_transforms = v2.Compose([
    v2.ToImage(), 
    v2.ToDtype(torch.float32, scale=True)
])

output_root = Path("prw_outputs")
output_root.mkdir(parents=True, exist_ok=True)

### Load full-frame test images

In [27]:
gallery_annotations = []
for img_path, ann_path in tqdm(
        zip(frame_dataset.image_paths, frame_dataset.annotations),
        total=len(frame_dataset),
        desc="Loading frame annotations"
    ):
    # Get frame name in the format 'cX_sY_ZZZZZZ' from the image path
    img_name = img_path.stem
    pids, boxes_xywh = parse_annotations_file(ann_path)

    if len(boxes_xywh) == 0:
        boxes_xyxy = np.zeros((0, 4), dtype=np.float32)
        pids = np.zeros((0,), dtype=np.int64)
    else:
        boxes_xywh = np.asarray(boxes_xywh, dtype=np.float32)
        boxes_xyxy = np.stack(
            [
                boxes_xywh[:, 0],
                boxes_xywh[:, 1],
                boxes_xywh[:, 0] + boxes_xywh[:, 2],
                boxes_xywh[:, 1] + boxes_xywh[:, 3],
            ],
            axis=1,
        )
        pids = np.asarray(pids, dtype=np.int64)

    cam_id = int(img_name.split("_")[0][1])
    # Required dictionary structure for provided eval_function in prw_evaluation.py
    gallery_annotations.append( 
        {
            "img_name": img_name,
            "boxes": boxes_xyxy,
            "pids": pids,
            "cam_id": cam_id,
        }
    )


class GalleryDatasetSimple(Dataset):
    def __init__(self, annotations, img_prefix):
        self.annotations = annotations
        self.img_prefix = img_prefix

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, i):
        return self.annotations[i]

gallery_dataset = GalleryDatasetSimple(gallery_annotations, str(data_root / "frames"))

Loading frame annotations:   0%|          | 0/6112 [00:00<?, ?it/s]

Loaded 6112 full-frame images


### Detect pedestrians, crop, and save

In [30]:
gallery_crop_dir = output_root / "gallery_crops"
gallery_crop_dir.mkdir(parents=True, exist_ok=True)

gallery_dets = []
gallery_crop_paths = []
gallery_img_names = []

for ann in tqdm(gallery_dataset.annotations, desc="Detect + crop"):
    img_name = ann["img_name"]
    img_path = data_root / "frames" / f"{img_name}.jpg"
    img_pil = Image.open(img_path).convert("RGB")

    img_t = frame_transforms(img_pil).unsqueeze(0).to(device)
    boxes, scores = detect_pedestrians(img_t, detector)

    if len(boxes) == 0:
        det = np.zeros((0, 5), dtype=np.float32)
        crop_paths = []
    else:
        det = np.hstack([boxes, scores[:, None]]).astype(np.float32)
        crops = crop_detections_from_frame(img_pil, boxes, output_size=detection_size)

        crop_paths = []
        for j, crop in enumerate(crops):
            crop_path = gallery_crop_dir / f"{img_name}_det{j:04d}.jpg"
            crop.save(crop_path)
            crop_paths.append(crop_path)

    gallery_dets.append(det)
    gallery_crop_paths.append(crop_paths)
    gallery_img_names.append(img_name)

Detect + crop:   0%|          | 0/6112 [00:00<?, ?it/s]

### Compute gallery embeddings

In [31]:
gallery_emb_dir = output_root / "gallery_embeddings"
gallery_emb_dir.mkdir(parents=True, exist_ok=True)

gallery_feats = []
for img_name, crop_paths in tqdm(zip(gallery_img_names, gallery_crop_paths), total=len(gallery_img_names), desc="Embedding gallery crops"):
    feats = []
    for crop_path in crop_paths:
        crop = Image.open(crop_path).convert("RGB")
        crop_t = detection_transforms(crop).unsqueeze(0).to(device)
        with torch.no_grad():
            emb = embedder(crop_t).cpu().numpy().reshape(-1)
        feats.append(emb)

    if len(feats) == 0:
        feats_arr = np.zeros((0, emb_size), dtype=np.float32)
    else:
        feats_arr = np.vstack(feats).astype(np.float32)

    gallery_feats.append(feats_arr)
    np.save(gallery_emb_dir / f"{img_name}.npy", feats_arr)

Embedding gallery crops:   0%|          | 0/6112 [00:00<?, ?it/s]

### Load query boxes and compute embeddings

In [32]:
class QueryDataset(Dataset):
    def __init__(
        self,
        data_root: Path,
        transforms: Optional[Callable] = None
    ) -> None:
        self.data_root = Path(data_root)
        self.transforms = transforms
        self.ext_images = "jpg"

        path_test_crops = self.data_root / "query_box"
        path_query_info = self.data_root / "query_info.txt"

        self.crops: List[Image.Image] = []
        self.ids: List[int] = []
        self.frame_references: List[str] = []
        self.boxes: List[List[float]] = []

        with open(path_query_info, "r") as query_info_file:
            lines = path_query_info.read_text().splitlines()
            for query_info in tqdm(lines, desc="Loading query boxes"):
                str_tokens = query_info.split()
                if len(str_tokens) < 6:
                    continue
                id_ped = int(str_tokens[0])
                box = [
                    float(str_tokens[1]),
                    float(str_tokens[2]),
                    float(str_tokens[3]),
                    float(str_tokens[4])
                ]
                box = [box[0], box[1], box[0] + box[2], box[1] + box[3]]
                frame_reference = str_tokens[5]

                self.ids.append(id_ped)
                self.frame_references.append(frame_reference)
                self.boxes.append(box)

                image_path = path_test_crops / f"{id_ped}_{frame_reference}.{self.ext_images}"
                crop = Image.open(image_path).convert("RGB").resize(
                    (detection_size[0], detection_size[1]),
                    Image.BILINEAR
                )
                self.crops.append(crop)

        # Build the annotations expected by eval_search_prw
        self.annotations = []
        for id_ped, box, frame_reference in zip(self.ids, self.boxes, self.frame_references):
            cam_id = int(frame_reference.split("_")[0][1])
            self.annotations.append(
                {
                    "img_name": frame_reference,
                    "boxes": np.array([box], dtype=np.float32),
                    "pids": int(id_ped),
                    "cam_id": cam_id,
                }
            )

    def __getitem__(self, idx):
        crop = self.crops[idx]
        if self.transforms is not None:
            crop = self.transforms(crop)

        return crop, self.ids[idx]

    def __len__(self):
        return len(self.crops)

In [33]:
query_dataset = QueryDataset(
    data_root=data_root,
    transforms=detection_transforms
)

query_embeddings = []
for idx in tqdm(range(len(query_dataset)), desc="Embedding query boxes"):
    crop_t, _ = query_dataset[idx]
    crop_t = crop_t.unsqueeze(0).to(device)
    with torch.no_grad():
        emb = embedder(crop_t).cpu().numpy().reshape(-1)
    query_embeddings.append(emb)

if len(query_embeddings) == 0:
    query_box_feats = np.zeros((0, emb_size), dtype=np.float32)
else:
    query_box_feats = np.vstack(query_embeddings).astype(np.float32)

np.save(output_root / "query_embeddings.npy", query_box_feats)

Loading query boxes:   0%|          | 0/2057 [00:00<?, ?it/s]

Embedding query boxes:   0%|          | 0/2057 [00:00<?, ?it/s]

# 5) Evaluation

## Evalutation metric computation

This final evaluation step provides us of two important metrics, `mAp` and `top-1 rank`, to get an quantitative measure of how good these results are and enabling direct and fast comparison with literature work.

In [34]:
det_thresh=0.20

ret = eval_search_prw(
    gallery_dataset=gallery_dataset,
    query_dataset=query_dataset,
    gallery_dets=gallery_dets,
    gallery_feats=gallery_feats,
    query_box_feats=query_box_feats,
    det_thresh=det_thresh,
    ignore_cam_id=True
)

search ranking:
  mAP = 23.69%
  top- 1 = 68.94%


## Visual evaluation

In [35]:
@interact(
    sample_index = widgets.IntSlider(min=0, max=len(gallery_dataset)-1, step=1, value=0, description="Gallery index"),
)
def plot_image_and_boxes(
    sample_index: int
) -> None:
    # GT
    sample_dict = gallery_dataset[sample_index]
    img_name = sample_dict['img_name']
    boxes = sample_dict['boxes']

    img_path = data_root / 'frames' / (img_name + '.jpg')
    img_pil = Image.open(img_path).convert('RGB')

    
    gt_with_bb = draw_boxes(
        img_pil,
        boxes=boxes,
        pids=[1] * len(boxes), # detector ignores ids
        scores=[1.0] * len(boxes),
    )
    

    # predicted

    sample_dict = gallery_dataset[sample_index]
    img_name = sample_dict['img_name']
    boxes_and_scores = gallery_dets[sample_index]
    # list comprehension to separate boxes and scores
    boxes = [box_score[:4] for box_score in boxes_and_scores]
    scores = [box_score[4] for box_score in boxes_and_scores]
    
    det_with_bb = draw_boxes(
        img_pil,
        boxes=boxes,
        pids=[1] * len(boxes), # detector ignores ids
        scores=scores
    )

    fig, axes = plt.subplots(1, 2, figsize=(18, 6))

    axes[0].imshow(gt_with_bb)
    axes[0].set_title("Ground truth bboxes")
    axes[0].axis("off")
    
    axes[1].imshow(det_with_bb)
    axes[1].set_title("Detected bboxes")
    axes[1].axis("off")

    fig.suptitle(f"Gallery image: {img_name}")
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=0, description='Gallery index', max=6111), Output()), _dom_classes=('wid…

In [36]:
@interact(
    idx1 = widgets.IntSlider(min=0, max=len(query_dataset)-1, step=1, value=0, description="Query index 1"),
    idx2 = widgets.IntSlider(min=0, max=len(query_dataset)-1, step=1, value=1, description="Query index 2")
)
def show_representations(idx1, idx2):
    def get_image_and_feat(idx):
        img, pid = query_dataset[idx]
        img = v2.ToPILImage()(img)

        features = np.asarray(query_box_feats[idx]).reshape(-1)
        if features.size != 2048:
            raise ValueError(f"Expected 2048-dim embedding, got {features.size}")
        feat_img = features.reshape(32, 64)
        return img, pid, features, feat_img

    img1, pid1, features1, feat_img1 = get_image_and_feat(idx1)
    _, axes = plt.subplots(1, 5, figsize=(18, 4))

    axes[0].set_title(f"Query image - Person ID1: {pid1}")
    axes[0].imshow(img1)
    axes[0].axis("off")

    axes[2].set_title(f"Embedding repr. - Person ID1: {pid1}")
    axes[2].imshow(feat_img1, cmap="viridis", aspect="auto", vmin=0, vmax=np.max(feat_img1))
    axes[2].axis("off")


    img2, pid2, features2, feat_img2 = get_image_and_feat(idx2)

    axes[1].set_title(f"Query image - Person ID2: {pid2}")
    axes[1].imshow(img2)
    axes[1].axis("off")

    axes[3].set_title(f"Embedding repr. - Person ID2: {pid2}")
    axes[3].imshow(feat_img2, cmap="viridis", aspect="auto", vmin=0, vmax=np.max(feat_img2))
    axes[3].axis("off")

    distance_matrix = (features1 - features2).reshape(32, 64)

    axes[4].set_title("Embedding repr. difference")
    axes[4].imshow(distance_matrix, cmap="Reds", aspect="auto", vmin=0, vmax=0.5)
    axes[4].axis("off")
    
    plt.tight_layout(rect=[0, 0, 1, 0.8])
    plt.show()


interactive(children=(IntSlider(value=0, description='Query index 1', max=2056), IntSlider(value=1, descriptio…

## Query-driven visual search
Pick a query crop and show the top-K most similar gallery detections (from `gallery_feats`) with color-coded correctness.

In [37]:
_gallery_cache = {}
_ann_by_img = None

def _to_numpy(value):
    if torch.is_tensor(value):
        return value.detach().cpu().numpy()
    return np.asarray(value)

def _get_ann_by_img():
    global _ann_by_img
    if _ann_by_img is None:
        _ann_by_img = {ann["img_name"]: ann for ann in gallery_dataset.annotations}
    return _ann_by_img

def _get_gt_box_for_pid(img_name, pid):
    ann = _get_ann_by_img().get(img_name)
    if ann is None:
        return None
    pids = _to_numpy(ann["pids"])
    match = np.where(pids == pid)[0]
    if match.size == 0:
        return None
    boxes = _to_numpy(ann["boxes"])
    return boxes[int(match[0])]

def _get_det_feat(i):
    cached = _gallery_cache.get(i)
    if cached is not None:
        return cached
    det = gallery_dets[i]
    feat = gallery_feats[i]
    _gallery_cache[i] = (det, feat)
    return det, feat

def _normalize_rows(mat):
    denom = np.linalg.norm(mat, axis=1, keepdims=True) + 1e-12
    return mat / denom

def _normalize_vec(vec):
    return vec / (np.linalg.norm(vec) + 1e-12)

def _draw_box(image, box, color, label=None, width=4):
    img = image.copy()
    painter = ImageDraw.Draw(img)
    x1, y1, x2, y2 = [float(v) for v in box]
    painter.rectangle([x1, y1, x2, y2], outline=color, width=width)

    if label:
        try:
            font = ImageFont.truetype(
                "/usr/share/fonts/truetype/liberation/LiberationMono-Regular.ttf",
                18
            )
        except Exception:
            font = ImageFont.load_default()

        text_w, text_h = font.getbbox(label)[-2:]
        text_x = max(0, x1)
        text_y = max(0, y1 - text_h)
        painter.rectangle(
            [text_x, text_y, text_x + text_w, text_y + text_h],
            fill=color
        )
        painter.text((text_x, text_y), label, fill="white", font=font)

    return img

In [38]:
@interact(
    query_index=widgets.IntSlider(
        min=0,
        max=len(query_dataset) - 1,
        step=1,
        value=0,
        description="Query index"
    ),
    top_k=widgets.IntSlider(
        min=1,
        max=10,
        step=1,
        value=3,
        description="Top K"
    )
)
def visualize_query_occurrences(query_index: int, top_k: int) -> None:
    ann = query_dataset.annotations[query_index]
    query_img_name = ann["img_name"]
    query_box = np.asarray(ann["boxes"]).reshape(-1)
    query_pid = int(ann["pids"])

    query_feat = np.asarray(query_box_feats[query_index]).reshape(-1)
    #query_feat = _normalize_vec(query_feat)

    query_crop_t, _ = query_dataset[query_index]
    query_crop = v2.ToPILImage()(query_crop_t)

    query_img_path = data_root / "frames" / f"{query_img_name}.jpg"
    query_img = Image.open(query_img_path).convert("RGB")
    query_img_vis = _draw_box(query_img, query_box, color="yellow", label="")

    # --- Plot 1: Query crop + query frame (unchanged) ---
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(query_crop)
    axes[0].set_title(f"Query crop (ID: {query_pid})")  # fixed missing f-string
    axes[0].axis("off")
    axes[1].imshow(query_img_vis)
    axes[1].set_title(f"Query frame: {query_img_name}")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

    # --- Retrieval ---
    det_thresh_local = det_thresh if "det_thresh" in globals() else 0.20
    results = []
    for i in range(len(gallery_dataset)):
        img_name = gallery_dataset.annotations[i]["img_name"]
        if img_name == query_img_name:
            continue

        det, feat = _get_det_feat(i)
        det = np.asarray(det)
        feat = np.asarray(feat)
        if det.size == 0 or feat.size == 0:
            continue
        if det.shape[0] != feat.shape[0]:
            min_n = min(det.shape[0], feat.shape[0])
            det = det[:min_n]
            feat = feat[:min_n]

        scores = det[:, 4]
        keep = np.where(scores >= det_thresh_local)[0]
        if keep.size == 0:
            continue
        det = det[keep]
        feat = feat[keep]

        feat = _normalize_rows(feat)
        sims = feat.dot(query_feat).ravel()
        for j, sim in enumerate(sims):
            results.append({
                "img_name": img_name,
                "box": det[j, :4],
                "score": float(det[j, 4]),
                "sim": float(sim),
            })

    if len(results) == 0:
        print("No detections found in gallery.")
        return

    results.sort(key=lambda x: x["sim"], reverse=True)
    results = results[:top_k]

    # --- Plot 2…N+1: one figure per result ---
    for pos, res in enumerate(results):
        img_name = res["img_name"]
        img_path = data_root / "frames" / f"{img_name}.jpg"
        img_pil = Image.open(img_path).convert("RGB")

        gt_box = _get_gt_box_for_pid(img_name, query_pid)
        if gt_box is None:
            is_correct = False
            iou_score = 0.0
        else:
            iou_thresh = 0.5
            iou_score = _compute_iou(res["box"], gt_box)
            is_correct = iou_score >= iou_thresh

        color = "lime" if is_correct else "red"
        label = "TP" if is_correct else "FP"
        img_vis = _draw_box(img_pil, res["box"], color=color, label="")

        # TODO: extract the crop using `crop_bounding_box`

        fig, ax = plt.subplots(1, 1, figsize=(6, 5))
        ax.imshow(img_vis)
        ax.set_title(
            f"({pos+1})  Label: {label}  |  Sim: {res['sim']:.3f}  |  IoU: {iou_score:.3f} | Frame: {img_name}",
            color=color,
            loc="left",
            pad=4.0,
        )
        ax.axis("off")
        plt.tight_layout()
        plt.show()

interactive(children=(IntSlider(value=0, description='Query index', max=2056), IntSlider(value=3, description=…